In [1]:
!pip install nltk scikit-learn pandas
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # You can change the model

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
df = pd.read_csv('twitter_training.csv')
df.rename(
    columns={
        list(df)[0]: 'TweetID',
        list(df)[1]: 'Entity',
        list(df)[2]: 'Sentiment',
        list(df)[3]: 'Tweet'
    },
    inplace=True
)
df.head()

,TweetID,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [4]:
df = df.drop(columns='TweetID')

In [5]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
def preprocess_text(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'[^\w\s]', '', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text) # This line requires 'punkt_tab'
    filtered_tokens = [stemmer.stem(word) for word in tokens if word.lower() not in stop_words]
    return ' '.join(filtered_tokens)

In [6]:
df['Processed_Tweet'] = df['Tweet'].apply(preprocess_text)

In [7]:
cv = CountVectorizer()
X = cv.fit_transform(df['Processed_Tweet'])  # Remove .toarray()
y = df['Sentiment']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [9]:
model = LogisticRegression(max_iter=1000) # Increased max_iter to avoid convergence warnings
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [12]:
def predict_sentiment(sentence):
    processed_sentence = preprocess_text(sentence)
    features = cv.transform([processed_sentence]).toarray()
    prediction = model.predict(features)[0]
    return prediction

In [18]:
new_sentence = "This is awsome!"
predicted_sentiment = predict_sentiment(new_sentence)
print(f"Sentiment of '{new_sentence}': {predicted_sentiment}")

Sentiment of 'This is awsome!': Positive
